In [2]:
!pip install kaggle-environments lion-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 26.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.6/175.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 31.1

In [3]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

### The Heuristic Bot (Expert Trajectory Generator)
This is the fully assembled heuristic bot integrating the column-assignment logic, the 4-day wheat start, and the Melon/Fertilizer transition.

In [4]:
import numpy as np

def get_season_phase(current_day):
    if current_day < 5: return "WHEAT_ACCUMULATION"
    elif current_day == 5: return "MELON_PIVOT_SETUP"
    elif 6 <= current_day <= 22: return "MELON_FERTILIZER_ENGINE"
    else: return "LIQUIDATION"

def heuristic_agent(obs, config):
    step = obs["step"]
    current_day = step // 24
    phase = get_season_phase(current_day)
    actions = []

    workers = obs.get("workers", [])
    bank = obs.get("bank", 0)
    inventory = obs.get("inventory", {})
    grid_width = obs.get("farm_width", 5)
    farm_grid = obs.get("farm_grid", [])

    if phase == "LIQUIDATION":
        for item, count in inventory.items():
            if count > 0 and "SEED" not in item:
                actions.append(f"SELL {item} {count}")
    else:
        for item in ["WHEAT", "MELON", "EGG"]:
            if inventory.get(item, 0) > 0:
                actions.append(f"SELL {item} {inventory[item]}")

        # Capped expansion at 10x10
        if bank >= 1000 and grid_width == 5: actions.append("BUY_LAND")

        if len(workers) < grid_width: actions.append("HIRE") #[cite: 1]

        if phase == "WHEAT_ACCUMULATION" and inventory.get("WHEAT_SEED", 0) < len(workers):
            actions.append("BUY_SEED WHEAT 1") #[cite: 2]
        elif phase == "MELON_PIVOT_SETUP":
            if inventory.get("GOOSE", 0) == 0 and not obs.get("has_goose", False):
                actions.append("BUY_ANIMAL GOOSE 1")
            if inventory.get("MELON_SEED", 0) < grid_width * 2:
                actions.append("BUY_SEED MELON 1")
        elif phase == "MELON_FERTILIZER_ENGINE":
            if inventory.get("MELON_SEED", 0) < len(workers): actions.append("BUY_SEED MELON 1")
            if inventory.get("WHEAT", 0) < 2: actions.append("BUY_PRODUCT WHEAT 1")

    for i, worker in enumerate(workers):
        wid = worker["id"]
        cx, cy = worker["x"], worker["y"]
        tx = i % grid_width #[cite: 1]

        if cx > tx: actions.append(f"{wid} WEST"); continue #[cite: 1]
        elif cx < tx: actions.append(f"{wid} EAST"); continue #[cite: 1]

        tile = farm_grid[cy][cx]
        if tile.get("can_harvest", False): actions.append(f"{wid} HARVEST"); continue

        if i == 0:
            if phase == "MELON_PIVOT_SETUP" and tile.get("is_empty", False) and cy == 0:
                actions.append(f"{wid} BUILD_COOP"); continue
            if tile.get("has_coop", False) and not tile.get("has_animal", False) and inventory.get("GOOSE", 0) > 0:
                actions.append(f"{wid} PLACE GOOSE 1"); continue
            if tile.get("has_animal", False):
                if tile.get("needs_feed", False): actions.append(f"{wid} FEED"); continue
                if tile.get("has_fertilizer", False): actions.append(f"{wid} COLLECT_FERTILIZER"); continue

        if tile.get("needs_water", False): actions.append(f"{wid} WATER"); continue
        if tile.get("is_melon", False) and tile.get("age", 0) >= 6 and not tile.get("is_fertilized", False) and inventory.get("FERTILIZER", 0) > 0:
            actions.append(f"{wid} FERTILIZE"); continue

        if tile.get("is_empty", False):
            if phase == "WHEAT_ACCUMULATION" and inventory.get("WHEAT_SEED", 0) > 0: actions.append(f"{wid} PLANT"); continue #[cite: 2]
            elif phase in ["MELON_PIVOT_SETUP", "MELON_FERTILIZER_ENGINE"] and inventory.get("MELON_SEED", 0) > 0:
                actions.append(f"{wid} PLANT"); continue

        # Capped spatial sweep at y=9
        max_y = 4 if grid_width == 5 else 9
        if cy < max_y: actions.append(f"{wid} SOUTH") #[cite: 1]
        else: actions.append(f"{wid} NORTH") #[cite: 1]

    return actions

### Neural Network & Lion Optimizer Setup
This defines the Actor-Critic model and initializes the Lion optimizer with the aggressive weight decay required for PPO.

In [5]:
import torch
import torch.nn as nn
from lion_pytorch import Lion

class KaggricultureAgent(nn.Module):
    def __init__(self, spatial_channels, global_features, action_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(spatial_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(global_features, 64),
            nn.ReLU()
        )
        # Updated for maximum expanded grid of 10x10
        combined_size = (64 * 10 * 10) + 64

        self.actor = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, spatial_x, global_x):
        # spatial_x shape: (B, C, H, W)
        conv_out = self.conv(spatial_x)
        mlp_out = self.mlp(global_x)
        combined = torch.cat([conv_out, mlp_out], dim=1)
        return self.actor(combined), self.critic(combined)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KaggricultureAgent(spatial_channels=5, global_features=44, action_dim=27).to(device)
optimizer = Lion(model.parameters(), lr=1e-5, weight_decay=1e-2)

### The Vectorized CuPy Engine
This module pushes high-throughput operations directly on the GPU, avoiding the slow CPU python dictionary iterations. It uses flat arrays to handle massive valid pair masking and state transitions simultaneously.

In [6]:
import cupy as cp
import torch

def cupy_to_torch(cp_array):
    return torch.from_dlpack(cp_array)

def torch_to_cupy(th_tensor):
    return cp.from_dlpack(th_tensor.contiguous())

class VectorizedKaggriculture:
    def __init__(self, num_envs=4096, grid_size=10, max_workers=5):
        self.num_envs = num_envs
        self.grid_size = grid_size
        self.max_workers = max_workers
        self.last_final_banks = cp.full((self.num_envs,), 3000.0, dtype=cp.float32)

        self.ID_EMPTY = 0
        self.ID_WHEAT = 1
        self.ID_CARROT = 2
        self.ID_TOMATO = 3
        self.ID_STRAWBERRY = 4
        self.ID_MELON = 5
        self.ID_COOP = 6
        self.ID_PASTURE = 7

    def reset(self):
        self.workers = cp.zeros((self.num_envs, self.max_workers, 2), dtype=cp.int32)

        # 5 Channels: 0=CropID, 1=Age, 2=Watered, 3=Fertilized, 4=Worker Position
        self.grids = cp.zeros((self.num_envs, 5, self.grid_size, self.grid_size), dtype=cp.float32)
        self.grids[:, 4, 0, 0] = 1.0

        # 45 Global Features (Index 44 is the Season Clock)
        self.globals = cp.zeros((self.num_envs, 45), dtype=cp.float32)
        self.globals[:, 0] = 3000.0
        self.globals[:, 18] = 1.0
        self.globals[:, 36:44] = 0.0
        self.globals[:, 44] = 0.0 # Clock starts at Day 1

        base_prices = cp.array([
            5, 10, 15, 20, 25, 40, 25, 40, 60, 80, 200, 150, 300, 500, 50, 120, 250
        ], dtype=cp.float32)

        self.globals[:, 19:36] = base_prices
        self.step_count = cp.zeros((self.num_envs,), dtype=cp.int32)

        return cupy_to_torch(self.grids), cupy_to_torch(self.globals)

    def get_action_mask(self):
        """Returns a boolean tensor of valid actions for the current state."""
        env_arange = cp.arange(self.num_envs)
        wx = self.workers[:, 0, 0]
        wy = self.workers[:, 0, 1]

        current_tiles = self.grids[env_arange, 0, wy, wx]
        current_age = self.grids[env_arange, 1, wy, wx]
        water_status = self.grids[env_arange, 2, wy, wx]
        bank = self.globals[:, 0]

        # Initialize all 27 actions as False, open up valid ones
        mask = cp.zeros((self.num_envs, 27), dtype=cp.bool_)

        # 0-3: Movement (Always Valid)
        mask[:, 0:4] = True

        # 4: Water (Only on unwatered crop tiles)
        is_crop = (current_tiles >= self.ID_WHEAT) & (current_tiles <= self.ID_MELON)
        mask[:, 4] = is_crop & (water_status == 0)

        # 5: Harvest (Only on mature crops or mature infra)
        is_mature_crop = is_crop & (current_age >= current_tiles)
        is_mature_infra = ((current_tiles == self.ID_COOP) | (current_tiles == self.ID_PASTURE)) & (current_age >= 1.0)
        mask[:, 5] = is_mature_crop | is_mature_infra

        # 6: Plant (Only on empty dirt when owning seeds)
        is_empty = (current_tiles == self.ID_EMPTY)
        has_any_seed = cp.any(self.globals[:, 2:7] > 0, axis=1)
        mask[:, 6] = is_empty & has_any_seed
        seed_reserve = 50.0

        # 7: Buy Land (If afford cost + reserve and not maxed)
        land_level = self.globals[:, 18]
        land_cost = cp.where(land_level == 1.0, 1000.0,
                    cp.where(land_level == 2.0, 2000.0,
                    cp.where(land_level == 3.0, 4000.0, 999999.0)))
        mask[:, 7] = (land_level < 4.0) & (bank >= (land_cost + seed_reserve))

        # 8-9: Build Infrastructure (Empty dirt + Afford + Reserve)
        mask[:, 8] = is_empty & (bank >= (500.0 + seed_reserve))   # Coop
        mask[:, 9] = is_empty & (bank >= (1000.0 + seed_reserve))  # Pasture


        # 10-26: Market Actions
        has_coop = cp.any((self.grids[:, 0, :, :] == self.ID_COOP), axis=(1,2))
        has_pasture = cp.any((self.grids[:, 0, :, :] == self.ID_PASTURE), axis=(1,2))

        for action_idx in range(10, 27):
            inv_idx = action_idx - 9
            price_idx = inv_idx + 18

            # Buying (Seeds, Geese, Sheep, Cows)
            if (inv_idx <= 6) or (12 <= inv_idx <= 14):
                can_afford = bank >= self.globals[:, price_idx]
                if inv_idx == 12:    # Goose
                    mask[:, action_idx] = can_afford & has_coop
                elif inv_idx in (13, 14): # Sheep, Cow
                    mask[:, action_idx] = can_afford & has_pasture
                else:
                    mask[:, action_idx] = can_afford

            # Selling (Crops, Eggs, Wool, Milk)
            else:
                mask[:, action_idx] = self.globals[:, inv_idx] > 0

        return cupy_to_torch(mask).to(torch.bool)

    def step(self, actions_tensor, current_epoch):
        cp_actions = torch_to_cupy(actions_tensor)
        env_arange = cp.arange(self.num_envs)

        anneal_factor = max(0.1, 1.0 - (current_epoch / 250.0))
        plant_reward = 2.0 * anneal_factor
        water_reward = 0.5 * anneal_factor
        harvest_reward = 10.0 * anneal_factor

        step_rewards = cp.full((self.num_envs,), -0.01, dtype=cp.float32)

        # ---------------------------------------------------------
        # WORKER MOVEMENT
        # ---------------------------------------------------------
        wx = self.workers[:, 0, 0]
        wy = self.workers[:, 0, 1]

        self.grids[env_arange, 4, wy, wx] = 0.0

        move_n = (cp_actions == 0)
        move_s = (cp_actions == 1)
        move_e = (cp_actions == 2)
        move_w = (cp_actions == 3)

        new_wy = cp.clip(wy - move_n.astype(cp.int32) + move_s.astype(cp.int32), 0, self.grid_size - 1)
        new_wx = cp.clip(wx + move_e.astype(cp.int32) - move_w.astype(cp.int32), 0, self.grid_size - 1)

        self.workers[:, 0, 0] = new_wx
        self.workers[:, 0, 1] = new_wy
        wx, wy = new_wx, new_wy
        self.grids[env_arange, 4, wy, wx] = 1.0

        current_tiles = self.grids[env_arange, 0, wy, wx]
        current_age = self.grids[env_arange, 1, wy, wx]
        water_status = self.grids[env_arange, 2, wy, wx]

        # ---------------------------------------------------------
        # BOUNDARY MASK
        # ---------------------------------------------------------
        is_q1 = (wx < 5)  & (wy < 5)
        is_q2 = (wx >= 5) & (wy < 5)
        is_q3 = (wx < 5)  & (wy >= 5)
        is_q4 = (wx >= 5) & (wy >= 5)

        land_level = self.globals[env_arange, 18]
        in_bounds = (is_q1 & (land_level >= 1.0)) | \
                    (is_q2 & (land_level >= 2.0)) | \
                    (is_q3 & (land_level >= 3.0)) | \
                    (is_q4 & (land_level >= 4.0))

        # ---------------------------------------------------------
        # FARM ACTIONS (WATER, HARVEST, PLANT)
        # ---------------------------------------------------------
        # WATER
        water_attempt = (cp_actions == 4)
        is_crop_tile = (current_tiles > self.ID_EMPTY) & (current_tiles <= self.ID_MELON)
        valid_water = water_attempt & is_crop_tile & (water_status == 0) & in_bounds

        step_rewards += cp.where(valid_water, water_reward, 0.0)
        self.grids[env_arange, 2, wy, wx] = cp.where(valid_water, 1, water_status)

        # HARVEST
        harvest_attempt = (cp_actions == 5)
        is_coop_tile = (current_tiles == self.ID_COOP)
        is_pasture_tile = (current_tiles == self.ID_PASTURE)

        valid_crop_harvest = harvest_attempt & is_crop_tile & (current_age >= current_tiles) & in_bounds
        valid_coop_harvest = harvest_attempt & is_coop_tile & (current_age >= 1.0) & in_bounds
        valid_pasture_harvest = harvest_attempt & is_pasture_tile & (current_age >= 1.0) & in_bounds
        valid_any_harvest = valid_crop_harvest | valid_coop_harvest | valid_pasture_harvest

        step_rewards += cp.where(valid_any_harvest, harvest_reward, 0.0)

        harvested_crop_ids = cp.where(valid_crop_harvest, current_tiles, 0).astype(cp.int32)
        self.globals[env_arange[valid_crop_harvest], harvested_crop_ids[valid_crop_harvest] + 6] += 1
        self.grids[env_arange, 0, wy, wx] = cp.where(valid_crop_harvest, self.ID_EMPTY, current_tiles)

        self.globals[:, 15] += cp.where(valid_coop_harvest, self.globals[:, 12], 0.0) # Geese -> Eggs
        self.globals[:, 16] += cp.where(valid_pasture_harvest, self.globals[:, 13], 0.0) # Sheep -> Wool
        self.globals[:, 17] += cp.where(valid_pasture_harvest, self.globals[:, 14], 0.0) # Cows -> Milk

        self.grids[env_arange, 1, wy, wx] = cp.where(valid_any_harvest, 0.0, current_age)

        # PLANT
        plant_attempt = (cp_actions == 6)
        has_any_seed = (self.globals[:, 2:7] > 0).any(axis=1)
        valid_plant = plant_attempt & (current_tiles == self.ID_EMPTY) & has_any_seed & in_bounds

        step_rewards += cp.where(valid_plant, plant_reward, 0.0)

        plant_melon  = valid_plant & (self.globals[:, 6] > 0)
        plant_strawb = valid_plant & ~plant_melon & (self.globals[:, 5] > 0)
        plant_tomato = valid_plant & ~(plant_melon | plant_strawb) & (self.globals[:, 4] > 0)
        plant_carrot = valid_plant & ~(plant_melon | plant_strawb | plant_tomato) & (self.globals[:, 3] > 0)
        plant_wheat  = valid_plant & ~(plant_melon | plant_strawb | plant_tomato | plant_carrot) & (self.globals[:, 2] > 0)

        updated_tiles = current_tiles.copy()
        updated_tiles = cp.where(plant_wheat, self.ID_WHEAT, updated_tiles)
        updated_tiles = cp.where(plant_carrot, self.ID_CARROT, updated_tiles)
        updated_tiles = cp.where(plant_tomato, self.ID_TOMATO, updated_tiles)
        updated_tiles = cp.where(plant_strawb, self.ID_STRAWBERRY, updated_tiles)
        updated_tiles = cp.where(plant_melon, self.ID_MELON, updated_tiles)

        self.grids[env_arange, 0, wy, wx] = cp.where(valid_plant, updated_tiles, self.grids[env_arange, 0, wy, wx])

        self.globals[:, 6] -= cp.where(plant_melon, 1, 0)
        self.globals[:, 5] -= cp.where(plant_strawb, 1, 0)
        self.globals[:, 4] -= cp.where(plant_tomato, 1, 0)
        self.globals[:, 3] -= cp.where(plant_carrot, 1, 0)
        self.globals[:, 2] -= cp.where(plant_wheat, 1, 0)

        # ---------------------------------------------------------
        # REAL ESTATE EXPANSION
        # ---------------------------------------------------------
        buy_land_attempt = (cp_actions == 7)
        land_cost = cp.where(land_level == 1.0, 1000.0,
                    cp.where(land_level == 2.0, 2000.0,
                    cp.where(land_level == 3.0, 4000.0, 999999.0)))

        valid_buy_land = buy_land_attempt & (self.globals[:, 0] >= land_cost) & (land_level < 4.0)
        self.globals[:, 0] -= cp.where(valid_buy_land, land_cost, 0.0)
        self.globals[:, 18] += cp.where(valid_buy_land, 1.0, 0.0)

        expansion_bonus = cp.where(land_level == 1.0, 5.0,
                          cp.where(land_level == 2.0, 10.0,
                          cp.where(land_level == 3.0, 20.0, 0.0)))
        step_rewards += cp.where(valid_buy_land, expansion_bonus * anneal_factor, 0.0)

        # ---------------------------------------------------------
        # BUILD INFRASTRUCTURE
        # ---------------------------------------------------------
        build_coop_attempt = (cp_actions == 8)
        build_pasture_attempt = (cp_actions == 9)

        is_empty_for_build = (self.grids[env_arange, 0, wy, wx] == self.ID_EMPTY)
        valid_build_coop = build_coop_attempt & is_empty_for_build & in_bounds & (self.globals[:, 0] >= 500.0)
        valid_build_pasture = build_pasture_attempt & is_empty_for_build & in_bounds & (self.globals[:, 0] >= 1000.0)

        self.globals[:, 0] -= cp.where(valid_build_coop, 500.0, 0.0)
        self.globals[:, 0] -= cp.where(valid_build_pasture, 1000.0, 0.0)

        self.grids[env_arange, 0, wy, wx] = cp.where(valid_build_coop, self.ID_COOP, self.grids[env_arange, 0, wy, wx])
        self.grids[env_arange, 0, wy, wx] = cp.where(valid_build_pasture, self.ID_PASTURE, self.grids[env_arange, 0, wy, wx])

        step_rewards += cp.where(valid_build_coop | valid_build_pasture, 10.0 * anneal_factor, 0.0)

        # ---------------------------------------------------------
        # TOWN SHOP GENERATOR
        # ---------------------------------------------------------
        is_new_day = (self.step_count % 24 == 0)
        unlock_trigger = is_new_day & (self.step_count >= 72) & (cp.random.rand(self.num_envs) < 0.25)

        if cp.any(unlock_trigger):
            new_shops = cp.random.randint(0, 8, size=(self.num_envs,), dtype=cp.int32)
            one_hot_shops = cp.eye(8, dtype=cp.float32)[new_shops]
            self.globals[:, 36:44] = cp.where(unlock_trigger.reshape(-1, 1),
                                              cp.maximum(self.globals[:, 36:44], one_hot_shops),
                                              self.globals[:, 36:44])

       # ---------------------------------------------------------
        # MARKET ECONOMY
        # ---------------------------------------------------------
        is_market_action = (cp_actions >= 10) & (cp_actions <= 26)

        target_inv_idx = cp.clip(cp_actions - 9, 0, 17)
        target_price_idx = target_inv_idx + 18


        current_prices = cp.where(is_market_action, self.globals[env_arange, target_price_idx], 0)
        current_inv = cp.where(is_market_action, self.globals[env_arange, target_inv_idx], 0)


        has_coop = cp.any((self.grids[:, 0, :, :] == self.ID_COOP), axis=(1,2))
        has_pasture = cp.any((self.grids[:, 0, :, :] == self.ID_PASTURE), axis=(1,2))

        is_buy_seed = is_market_action & (target_inv_idx <= 6)
        is_buy_goose = is_market_action & (target_inv_idx == 12)
        is_buy_pasture_animal = is_market_action & ((target_inv_idx == 13) | (target_inv_idx == 14))

        valid_buy = (is_buy_seed & (self.globals[:, 0] >= current_prices)) | \
                    (is_buy_goose & (self.globals[:, 0] >= current_prices) & has_coop) | \
                    (is_buy_pasture_animal & (self.globals[:, 0] >= current_prices) & has_pasture)

        self.globals[env_arange[valid_buy], target_inv_idx[valid_buy]] += 1
        self.globals[:, 0] -= cp.where(valid_buy, current_prices, 0.0)

        is_sell = is_market_action & (((target_inv_idx >= 7) & (target_inv_idx <= 11)) | (target_inv_idx >= 15))
        valid_sell = is_sell & (current_inv > 0)

        surge_mult = cp.ones((self.num_envs, 18), dtype=cp.float32)
        for i in range(8): # Check indices 36 to 43 and apply to proper inventory sell targets
            target = [7, 8, 9, 10, 11, 15, 16, 17][i]
            surge_mult[:, target] = cp.where(self.globals[:, 36+i] == 1.0, 2.5, 1.0)


        revenue = cp.where(valid_sell, current_inv, 0) * (current_prices * surge_mult[env_arange, target_inv_idx])

        self.globals[:, 0] += revenue
        self.globals[env_arange[valid_sell], target_inv_idx[valid_sell]] = 0 # Empty inventory on sell

        # THE FIX: Massive early-game dopamine for hitting the SELL button.
        # This adds a 10x multiplier to sales that decays as the epoch approaches 250.
        early_game_boost = 0.5 * anneal_factor
        step_rewards += (revenue * (self.step_count / 720.0) * (0.05 + early_game_boost))



        # ---------------------------------------------------------
        # END OF DAY TICK
        # ---------------------------------------------------------
        self.step_count += 1
        self.globals[:, 44] = self.step_count / 720.0 # Neural Network Clock

        end_of_day = (self.step_count % 24 == 0)
        if cp.any(end_of_day):
            active_grids = self.grids[end_of_day]
            is_crop = (active_grids[:, 0, :, :] >= self.ID_WHEAT) & (active_grids[:, 0, :, :] <= self.ID_MELON)
            is_watered = (active_grids[:, 2, :, :] == 1)
            active_grids[:, 1, :, :] += (is_crop & is_watered)
            active_grids[:, 2, :, :] = 0

            is_infra = (active_grids[:, 0, :, :] == self.ID_COOP) | (active_grids[:, 0, :, :] == self.ID_PASTURE)
            active_grids[:, 1, :, :] = cp.where(is_infra, 1.0, active_grids[:, 1, :, :])

            self.grids[end_of_day] = active_grids
            self.globals[end_of_day, 19:36] *= cp.random.uniform(0.95, 1.05, size=(int(cp.sum(end_of_day)), 17), dtype=cp.float32)

        # ---------------------------------------------------------
        # EPISODE RESET & TERMINAL REWARD
        # ---------------------------------------------------------
        dones = (self.step_count >= 720)
        if cp.any(dones):
            self.last_final_banks[dones] = self.globals[dones, 0].copy()

            final_profit = self.globals[dones, 0] - 3000.0
            step_rewards[dones] += cp.where(final_profit < 0, final_profit * 0.05, final_profit * 0.1)

            self.step_count[dones] = 0
            self.grids[dones] = 0.0
            self.grids[dones, 4, 0, 0] = 1.0
            self.workers[dones] = 0
            self.globals[dones] = 0.0
            self.globals[dones, 0] = 3000.0
            self.globals[dones, 18] = 1.0
            self.globals[dones, 36:44] = 0.0
            self.globals[dones, 44] = 0.0 # Reset clock

            base_prices = cp.array([5, 10, 15, 20, 25, 40, 25, 40, 60, 80, 200, 150, 300, 500, 50, 120, 250], dtype=cp.float32)
            self.globals[dones, 19:36] = base_prices

        return cupy_to_torch(self.grids).to(device), cupy_to_torch(self.globals).to(device), cupy_to_torch(step_rewards).to(device), cupy_to_torch(dones).to(device)

### PPO Self-Play Loop
This loop executes the vectorized state transitions and optimizes the policy using Lion.

In [7]:
# @title
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# 1. Instantiate the environment
env = VectorizedKaggriculture(num_envs=4096, grid_size=10, max_workers=15)

# 2. Re-initialize the model to accept the 5th spatial channel and 45th global feature (the clock)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KaggricultureAgent(spatial_channels=5, global_features=45, action_dim=27).to(device)

# 3. Attach the optimizer
optimizer = Lion(model.parameters(), lr=1e-5, weight_decay=1e-2)

# 4. Cleanly reset the environment states for the new loop
spatial_states, global_states = env.reset()

print("Environment and Model successfully initialized!")

Environment and Model successfully initialized!


In [ ]:
import torch.nn.functional as F
import torch

num_envs = 4096

# Override the default grid_size of 15 with 10
env = VectorizedKaggriculture(num_envs=num_envs, grid_size=10)

epochs = 500
steps_per_epoch = 720 # This will be the rollout_length
gamma = 0.999
epsilon = 0.2

spatial_states, global_states = env.reset()
for epoch in range(epochs):

    log_probs, values, rewards, masks, actions = [], [], [], [], []
    spatial_history, global_history = [], [] # ADDED: Lists to store states

    # 1. Collect Rollouts
    for step in range(steps_per_epoch): # Use steps_per_epoch as rollout_length
        spatial_history.append(spatial_states)
        global_history.append(global_states)

        with torch.no_grad():
            # 1. Ask the network for raw logits
            logits, value = model(spatial_states, global_states)

            # 2. Get the valid action mask from the environment
            action_mask = env.get_action_mask().to(device)

            # 3. MASKING LOGIC: Override invalid logits with Negative Infinity
            # This mathematically forces the Softmax probability of invalid actions to 0.0
            logits = logits.masked_fill(~action_mask, -1e9)

            # 4. Sample the action safely
            dist = torch.distributions.Categorical(logits=logits)
            current_action = dist.sample() # Use a temporary variable for the action
            current_log_prob = dist.log_prob(current_action) # Use a temporary variable for log_prob

        # Step the environment
        next_spatial, next_global, current_reward, current_done = env.step(current_action, epoch) # Use current_action, epoch, and temp variables

        values.append(value.squeeze())
        log_probs.append(current_log_prob) # Append the temporary log_prob
        rewards.append(current_reward) # Append the temporary reward
        masks.append(1.0 - current_done.float()) # Append mask from temporary done
        actions.append(current_action) # Append the temporary action

        spatial_states = next_spatial
        global_states = next_global

    # 2. Advantage Estimation
    returns = []
    R = torch.zeros(num_envs, device=device)
    for r, m in zip(reversed(rewards), reversed(masks)):
        R = r + gamma * R * m
        returns.insert(0, R)

    # Stack into tensors of shape (steps, num_envs, ...)
    returns = torch.stack(returns)
    values = torch.stack(values)
    log_probs = torch.stack(log_probs)
    actions = torch.stack(actions)
    spatial_history = torch.stack(spatial_history)
    global_history = torch.stack(global_history)

    advantages = returns - values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # 3. PPO Update Step (Mini-Batched)
    # Reshape tensors from (steps, num_envs, ...) to (steps * num_envs, ...)
    b_spatial = spatial_history.view(-1, *spatial_history.shape[2:])
    b_global = global_history.view(-1, *global_history.shape[2:])
    b_actions = actions.view(-1)
    b_log_probs = log_probs.view(-1)
    b_advantages = advantages.view(-1)
    b_returns = returns.view(-1)

    # --- NEW HYPERPARAMETERS ---
    ppo_epochs = 4           # Number of times to iterate over the collected rollout
    mini_batch_size = 8192   # Adjust downwards (e.g., 4096) if you still get OOM errors
    batch_size = b_spatial.size(0)

    # Run multiple epochs over the collected data
    for ppo_epoch in range(ppo_epochs):
        # Shuffle the indices to break temporal correlations
        indices = torch.randperm(batch_size, device=device)

        # Iterate over the batch in smaller chunks
        for start in range(0, batch_size, mini_batch_size):
            end = start + mini_batch_size
            mb_indices = indices[start:end]

            # Slice the mini-batch
            mb_spatial = b_spatial[mb_indices]
            mb_global = b_global[mb_indices]
            mb_actions = b_actions[mb_indices]
            mb_log_probs = b_log_probs[mb_indices]
            mb_advantages = b_advantages[mb_indices]
            mb_returns = b_returns[mb_indices]

            # Forward pass on the mini-batch
            new_logits, new_values = model(mb_spatial, mb_global)
            new_dist = torch.distributions.Categorical(logits=new_logits)
            new_log_probs = new_dist.log_prob(mb_actions)

            # Policy Loss
            ratio = torch.exp(new_log_probs - mb_log_probs.detach())
            surr1 = ratio * mb_advantages
            surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * mb_advantages
            actor_loss = -torch.min(surr1, surr2).mean()

            # Value Loss & Entropy
            critic_loss = F.mse_loss(new_values.squeeze(), mb_returns)
            entropy_loss = new_dist.entropy().mean()

            # Total Loss
            total_loss = actor_loss + 0.5 * critic_loss - 0.1 * entropy_loss

            # Optimize
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

    # Move the print statement outside the mini-batch loop
    # Logging block
    if epoch % 10 == 0:
        avg_reward = torch.stack(rewards).sum(0).mean().item()

        # Read the most recent Day 30 completion scores
        avg_final_score = env.last_final_banks.mean().item()

        print(f"Epoch {epoch:03d} | Avg Step Reward: {avg_reward:.2f} | Day 30 Final Score: {avg_final_score:.2f} | Loss: {total_loss.item():.4f}")
    # Checkpointing block (same indentation)
    if epoch > 0 and epoch % 50 == 0:
        checkpoint_path = f"/content/drive/MyDrive/kaggriculture_lion_epoch_{epoch}.pth"

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': total_loss.item(),
        }, checkpoint_path)

        print(f"--> Checkpoint saved to Drive: {checkpoint_path}")

Epoch 000 | Avg Step Reward: 475.25 | Day 30 Final Score: 65.42 | Loss: 86797.3516
Epoch 010 | Avg Step Reward: -52.60 | Day 30 Final Score: 9.40 | Loss: 18840.7773
Epoch 020 | Avg Step Reward: 1948.08 | Day 30 Final Score: 1761.69 | Loss: 3036861.7500
Epoch 030 | Avg Step Reward: 448.14 | Day 30 Final Score: 90.50 | Loss: 221560.4375
Epoch 040 | Avg Step Reward: 594.92 | Day 30 Final Score: 272.75 | Loss: 551730.8750
Epoch 050 | Avg Step Reward: 524.11 | Day 30 Final Score: 227.67 | Loss: 342440.4375
--> Checkpoint saved to Drive: /content/drive/MyDrive/kaggriculture_lion_epoch_50.pth
Epoch 060 | Avg Step Reward: 606.87 | Day 30 Final Score: 353.26 | Loss: 423652.7188
Epoch 070 | Avg Step Reward: 730.02 | Day 30 Final Score: 583.15 | Loss: 667876.8125
Epoch 080 | Avg Step Reward: 799.72 | Day 30 Final Score: 652.72 | Loss: 692612.8750
Epoch 090 | Avg Step Reward: 856.68 | Day 30 Final Score: 772.82 | Loss: 756053.1250
Epoch 100 | Avg Step Reward: 719.45 | Day 30 Final Score: 651.89 | 

Restart trainning if Colab got cutoff

In [17]:
!pip install lion-pytorch
import torch
from lion_pytorch import Lion
import torch.nn.functional as F

# 1. Re-initialize the model to accept 44 global features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KaggricultureAgent(spatial_channels=5, global_features=45, action_dim=27).to(device)

# 2. Re-attach the optimizer (it must track the new model's parameters)
# Note: Assuming you imported the Lion optimizer earlier!
optimizer = Lion(model.parameters(), lr=1e-5, weight_decay=1e-2)

# 3. Cleanly reset the environment states for the new loop
# Note: env is not defined at this point if this cell is run independently, but it will be defined later.
spatial_states, global_states = env.reset()

# 2. Define the path to your most recent checkpoint
# Update '50' to whatever your highest saved epoch number is
checkpoint_path = "/content/drive/MyDrive/kaggriculture_lion_epoch_50.pth"

# 3. Load the checkpoint dictionary
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Successfully loaded checkpoint. Resuming from Epoch {start_epoch}...")

# 4. Re-initialize the environment
num_envs = 4096
env = VectorizedKaggriculture(num_envs=num_envs)

epochs = 500
steps_per_epoch = 128
gamma = 0.99
epsilon = 0.2

# 5. Start the loop from the restored epoch instead of 0
for epoch in range(epochs):
    spatial_states, global_states = env.reset()

    log_probs, values, rewards, masks, actions = [], [], [], [], []
    spatial_history, global_history = [], [] # ADDED: Lists to store states

    # 1. Collect Rollouts
    for step in range(steps_per_epoch):
        spatial_history.append(spatial_states)
        global_history.append(global_states)

        with torch.no_grad():
            action_logits, value = model(spatial_states, global_states)

        action_dist = torch.distributions.Categorical(logits=action_logits)
        action = action_dist.sample()

        # PASS THE CURRENT EPOCH HERE
        next_spatial, next_global, reward, done = env.step(action, epoch)

        values.append(value.squeeze())
        log_probs.append(action_dist.log_prob(action))
        rewards.append(reward)
        masks.append(1.0 - done.float())
        actions.append(action)

        spatial_states = next_spatial
        global_states = next_global

    # 2. Advantage Estimation
    returns = []
    R = torch.zeros(num_envs, device=device)
    for r, m in zip(reversed(rewards), reversed(masks)):
        R = r + gamma * R * m
        returns.insert(0, R)

    # Stack into tensors of shape (steps, num_envs, ...)
    returns = torch.stack(returns)
    values = torch.stack(values)
    log_probs = torch.stack(log_probs)
    actions = torch.stack(actions)
    spatial_history = torch.stack(spatial_history)
    global_history = torch.stack(global_history)

    advantages = returns - values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # 3. PPO Update Step (Mini-Batched)
    # Reshape tensors from (steps, num_envs, ...) to (steps * num_envs, ...)
    b_spatial = spatial_history.view(-1, *spatial_history.shape[2:])
    b_global = global_history.view(-1, *global_history.shape[2:])
    b_actions = actions.view(-1)
    b_log_probs = log_probs.view(-1)
    b_advantages = advantages.view(-1)
    b_returns = returns.view(-1)

    # --- NEW HYPERPARAMETERS ---
    ppo_epochs = 4           # Number of times to iterate over the collected rollout
    mini_batch_size = 8192   # Adjust downwards (e.g., 4096) if you still get OOM errors
    batch_size = b_spatial.size(0)

    # Run multiple epochs over the collected data
    for ppo_epoch in range(ppo_epochs):
        # Shuffle the indices to break temporal correlations
        indices = torch.randperm(batch_size, device=device)

        # Iterate over the batch in smaller chunks
        for start in range(0, batch_size, mini_batch_size):
            end = start + mini_batch_size
            mb_indices = indices[start:end]

            # Slice the mini-batch
            mb_spatial = b_spatial[mb_indices]
            mb_global = b_global[mb_indices]
            mb_actions = b_actions[mb_indices]
            mb_log_probs = b_log_probs[mb_indices]

            # THE FIX: Change mb_ to b_ on the right side of the equals sign
            mb_advantages = b_advantages[mb_indices]
            mb_returns = b_returns[mb_indices]

            # Forward pass on the mini-batch
            new_logits, new_values = model(mb_spatial, mb_global)
            new_dist = torch.distributions.Categorical(logits=new_logits)
            new_log_probs = new_dist.log_prob(mb_actions)

            # Policy Loss
            ratio = torch.exp(new_log_probs - mb_log_probs.detach())
            surr1 = ratio * mb_advantages
            surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * mb_advantages
            actor_loss = -torch.min(surr1, surr2).mean()

            # Value Loss & Entropy
            critic_loss = F.mse_loss(new_values.squeeze(), mb_returns)
            entropy_loss = new_dist.entropy().mean()

            # Total Loss
            total_loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy_loss

            # Optimize
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

    # Move the print statement outside the mini-batch loop
    # Logging block
    if epoch % 10 == 0:
        avg_reward = torch.stack(rewards).sum(0).mean().item()

        # NEW: Track the actual average wealth across all 4,096 parallel farms
        avg_bank = global_states[:, 0].mean().item()

        print(f"Epoch {epoch:03d} | Avg RL Reward/Batch: {avg_reward:.2f} | Avg Bank Balance: {avg_bank:.2f} | Loss: {total_loss.item():.4f}")

    # Checkpointing block (same indentation)
    if epoch > 0 and epoch % 50 == 0:
        checkpoint_path = f"/content/drive/MyDrive/kaggriculture_lion_epoch_{epoch}.pth"

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': total_loss.item(),
        }, checkpoint_path)

        print(f"--> Checkpoint saved to Drive: {checkpoint_path}")

Successfully loaded checkpoint. Resuming from Epoch 51...
Epoch 000 | Avg RL Reward/Batch: 30.17 | Avg Bank Balance: 171.90 | Loss: 1411.0374
Epoch 010 | Avg RL Reward/Batch: 8.32 | Avg Bank Balance: 2500.00 | Loss: 0.3803
Epoch 020 | Avg RL Reward/Batch: 7.92 | Avg Bank Balance: 2500.00 | Loss: 0.3523
Epoch 030 | Avg RL Reward/Batch: 7.52 | Avg Bank Balance: 2500.00 | Loss: 0.3029


KeyboardInterrupt: 

### Download Trained Model


In [10]:
import torch
from google.colab import files

# 1. Save the model's weights to a file
save_path = '/content/drive/MyDrive/kaggriculture_lion.pth'
torch.save(model.state_dict(), save_path)

# 2. Trigger the browser to download the file
files.download(save_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import torch
import torch.nn as nn

# Define KaggricultureAgent class for this cell if it's meant to be self-contained for submission
class KaggricultureAgent(nn.Module):
    def __init__(self, spatial_channels, global_features, action_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(spatial_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(global_features, 64),
            nn.ReLU()
        )
        combined_size = (64 * 10 * 10) + 64 # Assuming grid_size=10

        self.actor = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, spatial_x, global_x):
        conv_out = self.conv(spatial_x)
        mlp_out = self.mlp(global_x)
        combined = torch.cat([conv_out, mlp_out], dim=1)
        return self.actor(combined), self.critic(combined)

# Initialize the exact same architecture used in training
model = KaggricultureAgent(spatial_channels=5, global_features=45, action_dim=27)

# Load the weights from the Kaggle input directory
model_path = '/content/drive/MyDrive/kaggriculture_lion.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# Set to evaluation mode
model.eval()

def agent(obs, config):
    # This part needs to parse the Kaggle 'obs' dictionary into spatial and global tensors
    # For demonstration, assume a simple parsing (this needs actual implementation based on Kaggle env format)
    # Example of parsing (requires actual obs structure):
    # farm_grid = np.array(obs['farm_grid']).transpose(2, 0, 1) # Assuming C, H, W order
    # spatial_state = torch.tensor(farm_grid, dtype=torch.float32).unsqueeze(0)
    # global_state = torch.tensor([obs['bank'], ...], dtype=torch.float32).unsqueeze(0)

    # Placeholder parsing for now - YOU NEED TO REPLACE THIS WITH ACTUAL OBS PARSING
    # The environment produces 5 channels, grid_size=10, global_features=44
    spatial_state = torch.zeros(1, 5, 10, 10)
    global_state = torch.zeros(1, 44)

    with torch.no_grad():
        # Pass tensors to the model
        action_logits, _ = model(spatial_state, global_state)

    # Decode logits to Kaggle string actions
    action_idx = torch.argmax(action_logits, dim=1).item()

    # Example mapping (needs to be consistent with your action_dim=27 mapping):
    # Replace with your actual mapping from action_idx to string commands.
    actions_map = {
        0: "NORTH", 1: "SOUTH", 2: "EAST", 3: "WEST", 4: "WATER",
        5: "HARVEST", 6: "PLANT", 7: "BUY_LAND", 8: "BUILD_COOP", 9: "BUILD_PASTURE",
        10: "BUY_SEED WHEAT 1", 11: "BUY_SEED CARROT 1", 12: "BUY_SEED TOMATO 1",
        13: "BUY_SEED STRAWBERRY 1", 14: "BUY_SEED MELON 1", 15: "BUY_ANIMAL GOOSE 1",
        16: "BUY_ANIMAL SHEEP 1", 17: "BUY_ANIMAL COW 1", 18: "SELL WHEAT 1",
        19: "SELL CARROT 1", 20: "SELL TOMATO 1", 21: "SELL STRAWBERRY 1",
        22: "SELL MELON 1", 23: "SELL EGG 1", 24: "SELL WOOL 1", 25: "SELL MILK 1",
        26: "PASS"
    }
    # This is a simplification; a full agent would need to handle worker IDs and specific actions.
    # For single worker, a single action is chosen.
    # For a multi-worker setup, you'd need to predict actions for each worker.

    # For a simple submission agent, assuming a single worker action if the RL agent handles it this way.
    return [actions_map.get(action_idx, "PASS")]

### Local Testing Enviornment
Test the trained agent locally using the kaggle-environments package to run a full match and generate a visual HTML replay.

In [15]:
!pip install kaggle-environments

In [20]:

import torch
import torch.nn as nn
from kaggle_environments import make
import numpy as np

# 1. Define your model architecture (must match exactly what you trained)
class KaggricultureAgent(torch.nn.Module):
    def __init__(self, spatial_channels, global_features, action_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(spatial_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(global_features, 64),
            nn.ReLU()
        )
        # Updated for maximum expanded grid of 10x10
        combined_size = (64 * 10 * 10) + 64

        self.actor = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, spatial_x, global_x):
        # spatial_x shape: (B, C, H, W)
        conv_out = self.conv(spatial_x)
        mlp_out = self.mlp(global_x)
        combined = torch.cat([conv_out, mlp_out], dim=1)
        return self.actor(combined), self.critic(combined)

# 2. Load the trained weights
device = torch.device("cpu") # Local testing is usually fine on CPU
model = KaggricultureAgent(spatial_channels=5, global_features=45, action_dim=27).to(device)

# THE FIX: Load the dictionary first, then extract just the model weights
checkpoint = torch.load('/content/drive/MyDrive/kaggriculture_lion_epoch_50.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

# 3. Define the Kaggle-compatible agent function
def trained_agent(obs, config):
    # ---------------------------------------------------------
    # A. SIM-TO-REAL PARSER (Translate JSON to Tensors)
    # ---------------------------------------------------------
    spatial = np.zeros((5, 10, 10), dtype=np.float32)
    global_feat = np.zeros((45,), dtype=np.float32)

    # 1. Parse The Clock
    step = obs.get('step', 0)
    global_feat[44] = step / 720.0

    # 2. Parse the Bank
    global_feat[0] = obs.get('bank', 3000.0)

    # 3. Parse Inventory
    inv = obs.get('inventory', {})
    inv_map = {
        "WHEAT_SEED": 2, "CARROT_SEED": 3, "TOMATO_SEED": 4, "STRAWBERRY_SEED": 5, "MELON_SEED": 6,
        "WHEAT": 7, "CARROT": 8, "TOMATO": 9, "STRAWBERRY": 10, "MELON": 11,
        "GOOSE": 12, "SHEEP": 13, "COW": 14,
        "EGG": 15, "WOOL": 16, "MILK": 17
    }
    for item_name, idx in inv_map.items():
        global_feat[idx] = inv.get(item_name, 0)

    # 4. Parse Land Level (Approximate based on farm grid size)
    farm_width = obs.get('farm_width', 5)
    farm_height = obs.get('farm_height', 5)
    area = farm_width * farm_height
    if area <= 25: global_feat[18] = 1.0   # Q1
    elif area <= 50: global_feat[18] = 2.0 # Q1 + Q2
    elif area <= 75: global_feat[18] = 3.0 # Q1 + Q2 + Q3
    else: global_feat[18] = 4.0            # Full 10x10

    # 5. Parse Prices (Using the base prices your env trained on)
    base_prices = [5, 10, 15, 20, 25, 40, 25, 40, 60, 80, 200, 150, 300, 500, 50, 120, 250]
    global_feat[19:36] = base_prices

    # 6. Parse the Spatial Grid
    farm_grid = obs.get('farm_grid', [])
    for cy in range(len(farm_grid)):
        for cx in range(len(farm_grid[cy])):
            if cy >= 10 or cx >= 10: continue

            tile = farm_grid[cy][cx]

            # Channel 0: Tile ID (Match training env IDs exactly)
            tile_id = 0
            if tile.get('is_wheat'): tile_id = 1
            elif tile.get('is_carrot'): tile_id = 2
            elif tile.get('is_tomato'): tile_id = 3
            elif tile.get('is_strawberry'): tile_id = 4
            elif tile.get('is_melon'): tile_id = 5
            elif tile.get('has_coop'): tile_id = 6
            elif tile.get('has_pasture'): tile_id = 7
            spatial[0, cy, cx] = tile_id

            # Channel 1: Age
            spatial[1, cy, cx] = tile.get('age', 0)

            # Channel 2: Water Status (1 if it has water, 0 if it needs it)
            spatial[2, cy, cx] = 0.0 if tile.get('needs_water', False) else 1.0

            # Channel 3: Fertilizer Status
            spatial[3, cy, cx] = 1.0 if tile.get('is_fertilized', False) else 0.0

    # Channel 4: Worker Position (Agent only trained on Worker 0)
    workers = obs.get('workers', [])
    worker_id = "w_0"
    if len(workers) > 0:
        worker_id = workers[0].get('id', 'w_0')
        wx, wy = workers[0].get('x', 0), workers[0].get('y', 0)
        if wx < 10 and wy < 10:
            spatial[4, wy, wx] = 1.0

    # ---------------------------------------------------------
    # B. NEURAL NETWORK INFERENCE
    # ---------------------------------------------------------
    # Convert numpy arrays to PyTorch tensors and add batch dimension (unsqueeze)
    spatial_tensor = torch.tensor(spatial, dtype=torch.float32).unsqueeze(0).to(device)
    global_tensor = torch.tensor(global_feat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        action_logits, _ = model(spatial_tensor, global_tensor)

    # Pick the greedy action (argmax) for testing, instead of sampling probabilities
    action_idx = torch.argmax(action_logits, dim=1).item()

    # ---------------------------------------------------------
    # C. ACTION TRANSLATION
    # ---------------------------------------------------------
    # Physical actions require the exact Worker ID prefix (e.g., "w_0 WATER")
    worker_actions = {
        0: "NORTH", 1: "SOUTH", 2: "EAST", 3: "WEST",
        4: "WATER", 5: "HARVEST", 6: "PLANT",
        8: "BUILD_COOP", 9: "BUILD_PASTURE"
    }

    # Market actions are global commands
    global_actions = {
        7: "BUY_LAND",
        10: "BUY_SEED WHEAT 1", 11: "BUY_SEED CARROT 1", 12: "BUY_SEED TOMATO 1",
        13: "BUY_SEED STRAWBERRY 1", 14: "BUY_SEED MELON 1", 15: "BUY_ANIMAL GOOSE 1",
        16: "BUY_ANIMAL SHEEP 1", 17: "BUY_ANIMAL COW 1", 18: "SELL WHEAT 1",
        19: "SELL CARROT 1", 20: "SELL TOMATO 1", 21: "SELL STRAWBERRY 1",
        22: "SELL MELON 1", 23: "SELL EGG 1", 24: "SELL WOOL 1", 25: "SELL MILK 1",
        26: "PASS"
    }

    # Route the output properly
    if action_idx in worker_actions:
        return [f"{worker_id} {worker_actions[action_idx]}"]
    else:
        return [global_actions.get(action_idx, "PASS")]

# 4. Define a simple opponent (or use your heuristic bot here)
def random_opponent(obs, config):
    return ["PASS"]

# 5. Run the local simulation
if __name__ == "__main__":
    # Initialize the environment
    env = make("kaggriculture", debug=True)

    # Run a full match (Turn 0 to 720)
    print("Running simulation...")
    steps = env.run([trained_agent, random_opponent])

    # 6. Render the output to view the match visually
    out_file = "/content/drive/MyDrive/local_test_replay.html"
    with open(out_file, "w") as f:
        f.write(env.render(mode="html"))

    print(f"Simulation complete. Open {out_file} in your web browser to watch the replay.")

    # Check final bank balances
    final_state = steps[-1]
    p1_bank = final_state[0]['observation'].get('bank', 0)
    p2_bank = final_state[1]['observation'].get('bank', 0)
    print(f"Final Score -> Trained Agent: {p1_bank} | Opponent: {p2_bank}")

Running simulation...
Simulation complete. Open /content/drive/MyDrive/local_test_replay.html in your web browser to watch the replay.
Final Score -> Trained Agent: 0 | Opponent: 0


### Submit Agent

In [29]:
%%writefile submission.py
import os
import torch
import torch.nn as nn
import numpy as np

# ---------------------------------------------------------
# 1. MODEL DEFINITION
# ---------------------------------------------------------
class KaggricultureAgent(nn.Module):
    def __init__(self, spatial_channels, global_features, action_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(spatial_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(global_features, 64),
            nn.ReLU()
        )
        combined_size = (64 * 10 * 10) + 64

        self.actor = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, spatial_x, global_x):
        conv_out = self.conv(spatial_x)
        mlp_out = self.mlp(global_x)
        combined = torch.cat([conv_out, mlp_out], dim=1)
        return self.actor(combined), self.critic(combined)

# ---------------------------------------------------------
# 2. GLOBAL INITIALIZATION (Runs once per match)
# ---------------------------------------------------------
device = torch.device("cpu")
model = None

def load_model():
    global model
    if model is None:
        model = KaggricultureAgent(spatial_channels=5, global_features=45, action_dim=27).to(device)
        base_dir = os.path.dirname(__file__)
        model_path = os.path.join(base_dir, 'model.pth')

        if not os.path.exists(model_path):
            model_path = 'model.pth'

        try:
            checkpoint = torch.load(model_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
        except Exception as e:
            print(f"Warning: Model weights not found. Error: {e}")

        model.eval()

# ---------------------------------------------------------
# 3. THE KAGGLE AGENT FUNCTION
# ---------------------------------------------------------
def agent(obs, config):
    load_model()

    spatial = np.zeros((5, 10, 10), dtype=np.float32)
    global_feat = np.zeros((45,), dtype=np.float32)

    step = obs.get('step', 0)
    global_feat[44] = step / 720.0
    global_feat[0] = obs.get('bank', 3000.0)

    inv = obs.get('inventory', {})
    inv_map = {
        "WHEAT_SEED": 2, "CARROT_SEED": 3, "TOMATO_SEED": 4, "STRAWBERRY_SEED": 5, "MELON_SEED": 6,
        "WHEAT": 7, "CARROT": 8, "TOMATO": 9, "STRAWBERRY": 10, "MELON": 11,
        "GOOSE": 12, "SHEEP": 13, "COW": 14,
        "EGG": 15, "WOOL": 16, "MILK": 17
    }
    for item_name, idx in inv_map.items():
        global_feat[idx] = inv.get(item_name, 0)

    farm_width = obs.get('farm_width', 5)
    farm_height = obs.get('farm_height', 5)
    area = farm_width * farm_height
    if area <= 25: global_feat[18] = 1.0
    elif area <= 50: global_feat[18] = 2.0
    elif area <= 75: global_feat[18] = 3.0
    else: global_feat[18] = 4.0

    base_prices = [5, 10, 15, 20, 25, 40, 25, 40, 60, 80, 200, 150, 300, 500, 50, 120, 250]
    global_feat[19:36] = base_prices

    farm_grid = obs.get('farm_grid', [])
    for cy in range(len(farm_grid)):
        for cx in range(len(farm_grid[cy])):
            if cy >= 10 or cx >= 10: continue
            tile = farm_grid[cy][cx]

            tile_id = 0
            if tile.get('is_wheat'): tile_id = 1
            elif tile.get('is_carrot'): tile_id = 2
            elif tile.get('is_tomato'): tile_id = 3
            elif tile.get('is_strawberry'): tile_id = 4
            elif tile.get('is_melon'): tile_id = 5
            elif tile.get('has_coop'): tile_id = 6
            elif tile.get('has_pasture'): tile_id = 7
            spatial[0, cy, cx] = tile_id

            spatial[1, cy, cx] = tile.get('age', 0)
            spatial[2, cy, cx] = 0.0 if tile.get('needs_water', False) else 1.0
            spatial[3, cy, cx] = 1.0 if tile.get('is_fertilized', False) else 0.0

    workers = obs.get('workers', [])
    worker_id = "w_0"
    if len(workers) > 0:
        worker_id = workers[0].get('id', 'w_0')
        wx, wy = workers[0].get('x', 0), workers[0].get('y', 0)
        if wx < 10 and wy < 10:
            spatial[4, wy, wx] = 1.0

    spatial_tensor = torch.tensor(spatial, dtype=torch.float32).unsqueeze(0).to(device)
    global_tensor = torch.tensor(global_feat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        action_logits, _ = model(spatial_tensor, global_tensor)

    action_idx = torch.argmax(action_logits, dim=1).item()

    worker_actions = {
        0: "NORTH", 1: "SOUTH", 2: "EAST", 3: "WEST",
        4: "WATER", 5: "HARVEST", 6: "PLANT",
        8: "BUILD_COOP", 9: "BUILD_PASTURE"
    }
    global_actions = {
        7: "BUY_LAND",
        10: "BUY_SEED WHEAT 1", 11: "BUY_SEED CARROT 1", 12: "BUY_SEED TOMATO 1",
        13: "BUY_SEED STRAWBERRY 1", 14: "BUY_SEED MELON 1", 15: "BUY_ANIMAL GOOSE 1",
        16: "BUY_ANIMAL SHEEP 1", 17: "BUY_ANIMAL COW 1", 18: "SELL WHEAT 1",
        19: "SELL CARROT 1", 20: "SELL TOMATO 1", 21: "SELL STRAWBERRY 1",
        22: "SELL MELON 1", 23: "SELL EGG 1", 24: "SELL WOOL 1", 25: "SELL MILK 1",
        26: "PASS"
    }

    if action_idx in worker_actions:
        return [f"{worker_id} {worker_actions[action_idx]}"]
    else:
        return [global_actions.get(action_idx, "PASS")]

Writing submission.py


In [30]:
import tarfile
import shutil

# 1. Copy the weights (I'm using your Epoch 100 checkpoint based on your previous logs)
# If you want to use a different epoch, just change the number in the filepath.
shutil.copyfile('/content/drive/MyDrive/kaggriculture_lion_epoch_100.pth', 'model.pth')

# 2. Create the Tar Archive
with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('submission.py', arcname='submission.py')
    tar.add('model.pth', arcname='model.pth')

print("Successfully packaged submission.tar.gz! Look in your Colab files menu on the left to download it.")

Successfully packaged submission.tar.gz! Look in your Colab files menu on the left to download it.
